# Week 5 companion - Sparse sensing and sparse dynamics
<!-- MIE690A article-aligned validation v4 -->
**Development companion:** local Run All is tested; the main-branch Colab link
becomes available only after this work is merged. This adds no new course week.

Learn to reconstruct a real wake using a few transverse-velocity sensors and
to identify an interpretable low-dimensional ODE. Prerequisites: Week 4/5 POD,
least squares and case-wise splitting. Allow 75-90 minutes; CPU only.
Lecture: `lectures/week05_modal_sensing.pdf`.

The ROI contains 32 x 78 fluid points, not the cylinder wall. Its coarse LBM
reference has 12 native cells per diameter and is not grid-independent CFD.
Filled contours interpolate level crossings only for display. No extra spatial
resolution, denoising or super-resolution is implied.
<!-- FLOWMLLAB_COLAB_LAUNCH_V1 -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ehsan-Roohi/FlowMLLab/blob/main/notebooks/week05_06/W5_Lab2_Sparse_Sensing_Dynamics.ipynb)


In [ ]:
# FLOWMLLAB_COLAB_BOOTSTRAP_V1
# In Colab this cell obtains the complete repository and installs the tested package.
# In a local checkout it leaves the active environment and working directory unchanged.
from pathlib import Path as _FlowMLLabPath
import os as _flowmllab_os
import subprocess as _flowmllab_subprocess
import sys as _flowmllab_sys

if "google.colab" in _flowmllab_sys.modules or _flowmllab_os.environ.get("COLAB_RELEASE_TAG"):
    _flowmllab_root = _FlowMLLabPath("/content/FlowMLLab")
    if not (_flowmllab_root / ".git").is_dir():
        _flowmllab_subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "https://github.com/Ehsan-Roohi/FlowMLLab.git", str(_flowmllab_root),
            ],
            check=True,
        )
    _flowmllab_subprocess.run(
        [
            _flowmllab_sys.executable, "-m", "pip", "install", "-q", "-e",
            f"{_flowmllab_root}[test]",
        ],
        check=True,
    )
    _flowmllab_notebook_dir = _flowmllab_root / "notebooks/week05_06"
    _flowmllab_os.chdir(_flowmllab_notebook_dir)
    for _flowmllab_path in (_flowmllab_root, _flowmllab_notebook_dir):
        if str(_flowmllab_path) not in _flowmllab_sys.path:
            _flowmllab_sys.path.insert(0, str(_flowmllab_path))
    print("FlowMLLab ready:", _flowmllab_root)

from pathlib import Path
import sys, hashlib, tempfile
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'flowmllab/modal_experiments.py').is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'qa'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from threadpoolctl import threadpool_limits
from flowmllab.modal_tools import *
from flowmllab.field_metrics import field_metrics, temporal_spectrum
from flowmllab.modal_experiments import PLAN, load_cases, forecast_experiment, sensor_experiment
from run_modal_labs import figures
tracked_evidence = [p for folder in ('data/modal_labs','results/modal_labs') for p in (ROOT/folder).rglob('*') if p.is_file()]
before = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in tracked_evidence}
cases = load_cases(ROOT/'data/modal_labs')
display(PLAN)
print('Checksummed author LBM; previously inspected cases, not a new blind CFD study.')

## 1. Warm-up: recover a known low-rank state
Fit $q=\bar q+\Phi a$ using training snapshots. Given observations
$y=Cq+\epsilon$, solve $(C\Phi)\hat a\simeq y-C\bar q$.
Pivoted QR selects $r$ independent rows; D-optimal greedy selection adds
oversampling rows. Arbitrary QR tail entries do not provide that criterion.
The pseudoinverse norm controls measurement-noise amplification; a full-rank
but ill-conditioned sensor matrix may still be poor.

First verify exact noiseless recovery on an explicitly rank-three control.
This synthetic control is a unit experiment; the next section uses actual LBM.

In [ ]:
rng = np.random.default_rng(42)
control = rng.normal(size=(30,3)) @ rng.normal(size=(3,40))
pod = fit_pod(control, rank=3)
sensor_ids = select_sensors(pod.modes, 6)
recovered = reconstruct_sensors(pod, sensor_ids, control[:,sensor_ids])
assert np.allclose(recovered, control, atol=1e-12)
print('Condition number:', np.linalg.cond(pod.modes[sensor_ids]))

## 2. Real-data protocol - freeze before execution
Training: complete Re90 and Re110 trajectories of v/U. Validation: Re100.
Retained test: Re105, previously inspected in other work, not a new blind case.
Fit an eight-mode POD and ALL sensor locations using training cases only.
Try 8, 16 and 32 sensors; select the smallest budget with mean optimized
validation relative L2 <= 5%, otherwise the best validation budget.

Compare five random layouts with optimized placement. Measurement noise is
Gaussian with standard deviation 1% of training v RMS, paired at coincident
locations. The five seeds describe artificial sensor noise/layout variation,
not independent CFD realizations. They do not support population confidence intervals.

The shared runner also performs the Week 7 forecast needed by the SINDy bridge;
all fits below are new and outputs go to a fresh temporary directory.

In [ ]:
# Fresh CPU fits, not cached retained predictions. Limit BLAS threads for reproducibility.
with threadpool_limits(limits=1):
    forecast, predictions = forecast_experiment(cases)
    sensing, examples = sensor_experiment(cases)
scratch = Path(tempfile.mkdtemp(prefix='flowmllab-modal-'))
figures(scratch, cases, forecast, predictions, sensing, examples)
print('Exploratory figure output:', scratch)

In [ ]:
rows = pd.DataFrame(sensing['records'])
display(rows[rows['split']=='validation'][['method','budget','seed','relative_l2']])
print('Validation-selected sensor budget:', sensing['selected_budget'])
display(rows[rows['split']=='test'][['method','budget','seed','condition_number','relative_l2','edge_relative_l2']])
display(Image(filename=str(scratch/'sensor_fields.png')))
display(Image(filename=str(scratch/'sensor_audit.png')))

## 3. Separate representation, observation and dynamics
The full-field POD oracle uses the complete target snapshot. It diagnoses
representation error and is not an operational sensor method. Compare methods
at equal sensor count. Explain why adding sensors cannot eliminate truncated modes.

For the dynamics bridge, use only Re110 frames 0:160 to fit a rank-two POD and
scale its coefficients. Build all monomials through degree three, integrate
their values by four-interval trapezoids, and regress coefficient increments.
Sequential thresholded least squares removes small terms and refits.
Validation chooses among thresholds 0.01, 0.05 and 0.1; the test continuation
must not alter this decision. There is no clipping or truth reset.

Inspect all candidates, including failed integrations, and the discovered
equations. A two-mode orbit cannot uniquely identify off-orbit cubic dynamics;
this is not discovery of Navier-Stokes. Its field error must be compared with
the **rank-two** oracle, not only with higher-rank models.

In [ ]:
display(pd.DataFrame(forecast['sindy_candidates']).T)
print('Selected sparse ODE:', forecast['selected_sindy'])
if forecast['selected_sindy']:
    for i, equation in enumerate(forecast['sindy_candidates'][forecast['selected_sindy']]['equations']):
        print(f'dz{i+1}/dt = {equation}')
display(pd.DataFrame({k:v['test'] for k,v in forecast['methods'].items()
    if k.startswith('SINDy') or k in ('DMD-r2','POD-r2-oracle')}).T)
display(Image(filename=str(scratch/'forecast_audit.png')))

## A shared metric contract (PDEBench-inspired, not identical scores)
Report $\|q-\hat q\|_{2,w}/\|q\|_{2,w}$, area-weighted RMSE, maximum error,
worst-frame relative error, ROI-edge error, and error of the supplied scalar integral.
An ROI edge is **not a physical wall**. An integral of vorticity or velocity is
**not automatically mass conservation**. Declare weights, units and geometry.
For zero reference norm, relative error is undefined (`None`), not epsilon-regularized.

Test a known offset before trusting the CFD score: adding one to a field of two
must give relative L2 = 0.5 and RMSE = 1. With 20 cells of area 0.25, the
absolute scalar-integral error must be 5. These are numerical identities, not fitted tolerances.

In [ ]:
truth = np.full((3,4,5), 2.0)
metrics = field_metrics(truth+1, truth, np.full((4,5), .25))
assert np.isclose(metrics['relative_l2'], .5)
assert np.isclose(metrics['rmse'], 1)
assert np.isclose(metrics['mean_absolute_scalar_integral_error'], 5)
assert field_metrics(np.ones_like(truth), 0*truth)['relative_l2'] is None
display(metrics)

## Sources, provenance and submission
The data are earlier **author-generated FlowMLLab LBM cases**, published in
[cylinder-cfd-v1](https://github.com/Ehsan-Roohi/FlowMLLab/releases/tag/cylinder-cfd-v1).
They are not copied package examples or data from the separate hypersonic DSMC article.
See `data/modal_labs/README.md` and its original/derived file hashes.

Original textbook implementations were inspired by
[PyDMD](https://github.com/PyDMD/PyDMD),
[PySensors](https://github.com/dynamicslab/pysensors),
[PySINDy](https://github.com/dynamicslab/pysindy), and
[PDEBench](https://github.com/pdebench/PDEBench).
No code or figures were copied; these are not wrappers, complete replacements,
or a claim of exact equivalence to the packages' advanced algorithms.

Submit the complete split, selected settings, all candidate/seed scores, one
failure explanation, and one proposed **new** validation experiment. Do not
retune on the retained test or overwrite reference evidence. Short CPU runtime
reflects reuse of already-generated CFD, not a fresh high-fidelity simulation.

In [ ]:
after = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in tracked_evidence}
assert after == before, 'Notebook modified retained data/evidence'
print('PASS: retained data and evidence unchanged.')